In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

try:
    import statsmodels.api as sm
    HAS_STATSMODELS = True
except Exception:
    HAS_STATSMODELS = False

plt.style.use("seaborn-v0_8")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

BASE_DIR = Path(".")
DATA_PATH = BASE_DIR / "data" / "spread" / "raw.csv"
EDA_DIR = BASE_DIR / "eda"
FIG_DIR = EDA_DIR / "figures"
TABLE_DIR = EDA_DIR / "tables"
PER_PAIR_DIR = EDA_DIR / "per_pair"

for d in [EDA_DIR, FIG_DIR, TABLE_DIR, PER_PAIR_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Input: {DATA_PATH}")
print(f"Output dir: {EDA_DIR.resolve()}")
print(f"Seaborn available: {HAS_SEABORN}")
print(f"Statsmodels available: {HAS_STATSMODELS}")

Input: data\spread\raw.csv
Output dir: C:\Users\vaibh\OneDrive\Documents\Projects\RL Pairs Trading\eda
Seaborn available: True
Statsmodels available: True


In [2]:
df = pd.read_csv(DATA_PATH, parse_dates=["Date"])

# Handle exported index columns from CSV writes
for c in ["Unnamed: 0", ""]:
    if c in df.columns:
        df = df.drop(columns=[c])

required_cols = ["Date", "Pair", "spread"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["spread"] = pd.to_numeric(df["spread"], errors="coerce")
df = df.sort_values(["Pair", "Date"]).reset_index(drop=True)

summary = pd.DataFrame(
    {
        "metric": ["rows", "pair_count", "date_min", "date_max", "missing_spread"],
        "value": [
            len(df),
            df["Pair"].nunique(),
            df["Date"].min().date(),
            df["Date"].max().date(),
            int(df["spread"].isna().sum()),
        ],
    }
)
summary.to_csv(TABLE_DIR / "dataset_summary.csv", index=False)

# Avoid hard dependency on tabulate
try:
    summary.to_markdown(TABLE_DIR / "dataset_summary.md", index=False)
except Exception:
    (TABLE_DIR / "dataset_summary.md").write_text(summary.to_string(index=False), encoding="utf-8")

summary

,metric,value
0,rows,24492
1,pair_count,12
2,date_min,2018-01-01
3,date_max,2026-04-08
4,missing_spread,2868


In [3]:
spread_clean = df["spread"].dropna()

global_stats = spread_clean.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_frame(name="spread")
global_stats.loc["skew"] = spread_clean.skew()
global_stats.loc["kurtosis"] = spread_clean.kurtosis()
global_stats.to_csv(TABLE_DIR / "global_spread_stats.csv")

fig, ax = plt.subplots(figsize=(10, 5))
if HAS_SEABORN:
    sns.histplot(spread_clean, bins=80, kde=True, ax=ax)
else:
    ax.hist(spread_clean, bins=80)
ax.set_title("Global Spread Distribution (Histogram + KDE)")
ax.set_xlabel("spread")
fig.tight_layout()
fig.savefig(FIG_DIR / "global_hist_kde.png", dpi=150)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if HAS_SEABORN:
    sns.boxplot(x=spread_clean, ax=axes[0])
    sns.violinplot(x=spread_clean, ax=axes[1])
else:
    axes[0].boxplot(spread_clean, vert=False)
    axes[1].hist(spread_clean, bins=80)
axes[0].set_title("Global Spread Boxplot")
axes[1].set_title("Global Spread Violin")
fig.tight_layout()
fig.savefig(FIG_DIR / "global_box_violin.png", dpi=150)
plt.close(fig)

if HAS_STATSMODELS:
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111)
    sm.qqplot(spread_clean, line="45", ax=ax)
    ax.set_title("QQ Plot - Global Spread")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "global_qqplot.png", dpi=150)
    plt.close(fig)

global_stats.head(12)

,spread
count,21624.000000
mean,0.755210
std,1.295652
min,-2.278395
1%,-2.120848
5%,-1.943116
25%,-0.215660
50%,0.932395
75%,1.727310
95%,2.554794


In [4]:
daily = (
    df.groupby("Date", as_index=False)["spread"]
    .agg(mean_spread="mean", median_spread="median", std_spread="std")
    .sort_values("Date")
)
daily["rolling_std_30d"] = daily["std_spread"].rolling(30, min_periods=5).mean()
daily.to_csv(TABLE_DIR / "daily_cross_sectional_spread_stats.csv", index=False)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(daily["Date"], daily["mean_spread"], label="Mean", linewidth=1.5)
axes[0].plot(daily["Date"], daily["median_spread"], label="Median", linewidth=1.2)
axes[0].set_title("Cross-Sectional Spread Trend Over Time")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(daily["Date"], daily["std_spread"], label="Daily Std", linewidth=1)
axes[1].plot(daily["Date"], daily["rolling_std_30d"], label="30D Rolling Std", linewidth=1.5)
axes[1].set_title("Cross-Sectional Spread Dispersion")
axes[1].legend()
axes[1].grid(alpha=0.25)

fig.tight_layout()
fig.savefig(FIG_DIR / "global_time_series.png", dpi=150)
plt.close(fig)

daily.head()

,Date,mean_spread,median_spread,std_spread,rolling_std_30d
0,2018-01-01,0.902878,1.223995,1.408835,NaN
1,2018-01-02,0.899921,1.215363,1.404052,NaN
2,2018-01-03,0.901246,1.216477,1.408032,NaN
3,2018-01-04,0.915134,1.231685,1.415908,NaN
4,2018-01-05,0.919155,1.244050,1.408272,1.40902


In [5]:
pair_stats = (
    df.groupby("Pair")["spread"]
    .agg(
        count="count",
        mean="mean",
        std="std",
        min="min",
        q01=lambda x: x.quantile(0.01),
        q05=lambda x: x.quantile(0.05),
        q25=lambda x: x.quantile(0.25),
        median="median",
        q75=lambda x: x.quantile(0.75),
        q95=lambda x: x.quantile(0.95),
        q99=lambda x: x.quantile(0.99),
        max="max",
        skew="skew",
        kurtosis=pd.Series.kurt,
    )
    .sort_values("std", ascending=False)
)

pair_stats.to_csv(TABLE_DIR / "per_pair_spread_stats.csv")

ranking_mean_high = pair_stats.sort_values("mean", ascending=False).head(20)
ranking_mean_low = pair_stats.sort_values("mean", ascending=True).head(20)
ranking_std_high = pair_stats.sort_values("std", ascending=False).head(20)

ranking_mean_high.to_csv(TABLE_DIR / "top20_pairs_by_mean_spread.csv")
ranking_mean_low.to_csv(TABLE_DIR / "bottom20_pairs_by_mean_spread.csv")
ranking_std_high.to_csv(TABLE_DIR / "top20_pairs_by_spread_std.csv")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ranking_mean_high["mean"].sort_values().plot(kind="barh", ax=axes[0])
axes[0].set_title("Top 20 Pairs by Mean Spread")
axes[0].set_xlabel("mean spread")
ranking_std_high["std"].sort_values().plot(kind="barh", ax=axes[1], color="tab:orange")
axes[1].set_title("Top 20 Pairs by Spread Volatility")
axes[1].set_xlabel("std spread")
fig.tight_layout()
fig.savefig(FIG_DIR / "cross_pair_rankings.png", dpi=150)
plt.close(fig)

pair_stats.head()

,count,mean,std,min,q01,q05,q25,median,q75,q95,q99,max,skew,kurtosis
Pair,,,,,,,,,,,,,,
PNB.NS|BANKBARODA.NS,2041,1.173709,0.289110,0.544244,0.575897,0.653572,0.970667,1.258938,1.376440,1.541529,2.027393,2.109153,-0.138647,-0.013842
PNB.NS|CANBK.NS,2041,1.275541,0.250899,0.708448,0.746521,0.809852,1.155694,1.296910,1.428076,1.634827,2.076940,2.157467,0.073021,0.707438
HINDPETRO.NS|IOC.NS,2041,2.586187,0.219687,2.046394,2.257410,2.321684,2.422875,2.517284,2.696777,3.012026,3.056661,3.103709,0.744777,-0.520655
UNIONBANK.NS|CANBK.NS,2041,0.181388,0.181330,-0.265191,-0.221359,-0.154322,0.076924,0.181092,0.293471,0.489072,0.610334,0.652213,-0.037669,-0.098037
VBL.NS|DOMS.NS,567,2.437244,0.170210,2.116322,2.151880,2.220479,2.282800,2.407445,2.606647,2.707310,2.751464,2.764038,0.280753,-1.284213


In [6]:
for pair, pair_df in df.groupby("Pair"):
    safe_pair = pair.replace("/", "_").replace("\\", "_").replace("|", "__")
    pair_dir = PER_PAIR_DIR / safe_pair
    pair_dir.mkdir(parents=True, exist_ok=True)

    pair_df = pair_df.sort_values("Date")
    pair_spread = pair_df["spread"].dropna()

    # Time-series plot
    fig, ax = plt.subplots(figsize=(11, 3.8))
    ax.plot(pair_df["Date"], pair_df["spread"], linewidth=1)
    ax.set_title(f"Spread Time Series: {pair}")
    ax.set_xlabel("Date")
    ax.set_ylabel("spread")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(pair_dir / "timeseries.png", dpi=140)
    plt.close(fig)

    # Distribution panel: histogram + boxplot
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    if HAS_SEABORN:
        sns.histplot(pair_spread, bins=45, kde=True, ax=axes[0])
        sns.boxplot(x=pair_spread, ax=axes[1])
    else:
        axes[0].hist(pair_spread, bins=45)
        axes[1].boxplot(pair_spread, vert=False)
    axes[0].set_title(f"Distribution: {pair}")
    axes[1].set_title(f"Boxplot: {pair}")
    fig.tight_layout()
    fig.savefig(pair_dir / "distribution.png", dpi=140)
    plt.close(fig)

print(f"Generated per-pair plots under: {PER_PAIR_DIR}")

Generated per-pair plots under: eda\per_pair


In [7]:
table_files = sorted([p.relative_to(EDA_DIR).as_posix() for p in TABLE_DIR.rglob("*") if p.is_file()])
figure_files = sorted([p.relative_to(EDA_DIR).as_posix() for p in FIG_DIR.rglob("*") if p.is_file()])
per_pair_dirs = sorted([p.name for p in PER_PAIR_DIR.iterdir() if p.is_dir()])

print("EDA export summary")
print("=" * 60)
print(f"Rows analyzed        : {len(df):,}")
print(f"Pairs analyzed       : {df['Pair'].nunique():,}")
print(f"Tables generated     : {len(table_files)}")
print(f"Figures generated    : {len(figure_files)}")
print(f"Per-pair folders     : {len(per_pair_dirs)}")
print("\nSample tables:")
for f in table_files[:10]:
    print(f" - {f}")
print("\nSample figures:")
for f in figure_files[:10]:
    print(f" - {f}")

EDA export summary
Rows analyzed        : 24,492
Pairs analyzed       : 12
Tables generated     : 8
Figures generated    : 5
Per-pair folders     : 12

Sample tables:
 - tables/bottom20_pairs_by_mean_spread.csv
 - tables/daily_cross_sectional_spread_stats.csv
 - tables/dataset_summary.csv
 - tables/dataset_summary.md
 - tables/global_spread_stats.csv
 - tables/per_pair_spread_stats.csv
 - tables/top20_pairs_by_mean_spread.csv
 - tables/top20_pairs_by_spread_std.csv

Sample figures:
 - figures/cross_pair_rankings.png
 - figures/global_box_violin.png
 - figures/global_hist_kde.png
 - figures/global_qqplot.png
 - figures/global_time_series.png
